## Legacy Access via Service Principal

Service Principal credentials are retrieved securely from the Azure Key Vault-backed Secret Scope.

The credentials are then used to configure OAuth authentication for direct access to the ADLS Gen2 container.

In [0]:
dbutils.secrets.list(scope="viktoriia_kalenichenko_scope")

In [0]:
client_id = dbutils.secrets.get(scope = "viktoriia_kalenichenko_scope", key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope = "viktoriia_kalenichenko_scope", key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope = "viktoriia_kalenichenko_scope", key="tenant-id")

In [0]:
storage_account = "dlsua5816bd"
container = "victoriya44250"

configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
source = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/"
)
display(dbutils.fs.ls(source))


**Result:** Direct access to the ADLS Gen2 container using Service Principal authentication was successful.

### Legacy DBFS Mount

In [0]:
mount_point = "/mnt/viktoriia44250_legacy"
try:
    dbutils.fs.mount(
        source=source,
        mount_point=mount_point,
        extra_configs=configs
    )
    print(f"Mounted successfully to {mount_point}")

except Exception as e:
    print(e)
    

**Result:** Calling `dbutils.fs.mount()` on the current Unity Catalog-enabled shared cluster fails with a `not whitelisted` exception.

This confirms that the legacy DBFS mount approach is restricted on this cluster configuration. Direct `abfss://` access through Service Principal authentication works successfully and should be preferred over legacy mounts in this environment.